# Day 07 Lab: Car License Plate Detection with YOLO

**Goal:** Train a YOLOv8 object detector to find license plates in car images.

**Dataset:** 433 `.png` images (346 train / 87 test) with YOLO-format labels: `class xc yc w h` (normalized). Derived from the Kaggle *Car License Plate Detection* dataset.

**Expected folder layout (put it next to this notebook):**
```
dataset/
├── train/
│   ├── images/   (346 .png)
│   └── labels/   (346 .txt)
├── test/
│   ├── images/   (87 .png)
│   └── labels/   (87 .txt)
└── data.yaml
```


## 1. Setup
Run once in VS Code's terminal, or uncomment the line below. Make sure the notebook kernel is the Python environment where you installed the packages.

In [ ]:
# !pip install ultralytics opencv-python matplotlib pyyaml

In [ ]:
import os, glob, random, yaml
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

random.seed(42)

## 2. Configure dataset path
Change `DATASET_DIR` if your folder has a different name or location.

In [ ]:
DATASET_DIR = Path("dataset").resolve()   # <-- edit if needed

TRAIN_IMG = DATASET_DIR / "train" / "images"
TRAIN_LBL = DATASET_DIR / "train" / "labels"
TEST_IMG  = DATASET_DIR / "test" / "images"
TEST_LBL  = DATASET_DIR / "test" / "labels"

for p in [TRAIN_IMG, TRAIN_LBL, TEST_IMG, TEST_LBL]:
    n = len(list(p.glob("*"))) if p.exists() else "MISSING"
    print(f"{str(p.relative_to(DATASET_DIR)):15s} -> {n} files")

## 3. Create a `data.yaml` with absolute paths
YOLO is picky about relative paths, so we write a fresh config that always works on your machine. The **test** folder is used for validation.

In [ ]:
data_cfg = {
    "path": str(DATASET_DIR),
    "train": "train/images",
    "val": "test/images",
    "nc": 1,
    "names": ["license_plate"],
}
YAML_PATH = DATASET_DIR / "data_local.yaml"
with open(YAML_PATH, "w") as f:
    yaml.safe_dump(data_cfg, f)
print(open(YAML_PATH).read())

## 4. Explore the data
Look at one label file, then draw the boxes on a few random training images.

In [ ]:
sample_lbl = random.choice(list(TRAIN_LBL.glob("*.txt")))
print(sample_lbl.name)
print(sample_lbl.read_text())
print("Format: class  x_center  y_center  width  height  (all normalized 0-1)")

In [ ]:
def draw_yolo_boxes(img_path, lbl_path):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            cls, xc, yc, bw, bh = map(float, line.split())
            x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
            x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
    return img

imgs = random.sample(list(TRAIN_IMG.glob("*.png")), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, p in zip(axes.ravel(), imgs):
    ax.imshow(draw_yolo_boxes(p, TRAIN_LBL / (p.stem + ".txt")))
    ax.set_title(p.name); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Train
`yolov8n.pt` (nano) is small and fast, ideal for a lab. It downloads automatically on first run.

- **GPU:** ~5 to 10 minutes for 30 epochs.
- **CPU only:** slower; lower `EPOCHS` to 10 to 15 or reduce `IMGSZ` to 416.

In [ ]:
EPOCHS = 30
IMGSZ  = 640
BATCH  = 16

model = YOLO("yolov8n.pt")
results = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project="runs_lab",
    name="plate_detector",
    exist_ok=True,
    workers=2,      # set to 0 if you get multiprocessing errors on Windows
)

## 6. Training curves and metrics

In [ ]:
RUN_DIR = Path("runs_lab/plate_detector")

img = cv2.cvtColor(cv2.imread(str(RUN_DIR / "results.png")), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(16, 8)); plt.imshow(img); plt.axis("off"); plt.show()

In [ ]:
best = YOLO(RUN_DIR / "weights" / "best.pt")
metrics = best.val(data=str(YAML_PATH))
print(f"Precision : {metrics.box.mp:.3f}")
print(f"Recall    : {metrics.box.mr:.3f}")
print(f"mAP@0.5   : {metrics.box.map50:.3f}")
print(f"mAP@0.5:95: {metrics.box.map:.3f}")

## 7. Predict on test images
Green = ground truth, red = model prediction.

In [ ]:
test_imgs = random.sample(list(TEST_IMG.glob("*.png")), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, p in zip(axes.ravel(), test_imgs):
    canvas = draw_yolo_boxes(p, TEST_LBL / (p.stem + ".txt"))   # ground truth (green)
    pred = best.predict(str(p), conf=0.25, verbose=False)[0]
    for box in pred.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(canvas, f"{box.conf[0]:.2f}", (x1, max(y1 - 8, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
    ax.imshow(canvas); ax.set_title(p.name); ax.axis("off")

plt.tight_layout(); plt.show()

## 8. Try your own image

In [ ]:
MY_IMAGE = "my_car.png"   # <-- put any car photo next to the notebook

if os.path.exists(MY_IMAGE):
    r = best.predict(MY_IMAGE, conf=0.25, verbose=False)[0]
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
else:
    print(f"'{MY_IMAGE}' not found, skipping.")

## 9. Lab questions
1. What are the precision, recall and mAP@0.5 on the test set? Is the model good enough?
2. Where does it fail (small plates, angled plates, night images)? Why might that be?
3. Change `EPOCHS` or `IMGSZ` and retrain. How do the metrics change?
4. **Challenge:** try `yolov8s.pt` instead of `yolov8n.pt` and compare accuracy versus training time.
